In [ ]:
# Last amended: 25th August, 2026

# RAG App Implementation Using Llama 3.2, Ollama & PostgreSQL

In [1]:
%reset -f

### Define the query and its embedding

In [ ]:
# 1.0 Call libraries
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings


In [ ]:
# 1.1
Settings.embed_model=None
Settings.llm = None

/home/ashok/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings have been explicitly disabled. Using MockEmbedding.
LLM is explicitly disabled. Using MockLLM.


In [ ]:
# 2.0

# 2.01 Initialize Ollama
Settings.llm = Ollama(model="mistral:latest", request_timeout=120.0)
# 2.02 bge-m3 gives problems
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text") 

In [ ]:
# 3.0 Initialise vector store
from llama_index.vector_stores.postgres import PGVectorStore

# 3.1
vector_store = PGVectorStore.from_params(
                                        database="ashok",
                                        host="192.240.1.27",
                                        password="ashok",
                                        port=5432,
                                        user="ashok",
                                        table_name="ollama_embeddings",
                                        embed_dim=768  # Match your embedding model's output
                                        )

In [ ]:
# 4.0
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext

In [ ]:
# 4.1 Load your local data
#     Each Document contains the text extracted from a file (such as a PDF, DOCX, or TXT) 
#     and associated metadata like filename or page number
documents = SimpleDirectoryReader("/home/ashok/lprojects/pdf").load_data()

In [ ]:
# 4.2 Link the vector store to the storage context
"""
A vector_store is a component that stores and retrieves embedding vectors for your data
(e.g., using PGVectorStore, ChromaVectorStore, PineconeVectorStore). It is responsible 
only for managing the embeddings and their metadata (module_guides/storing).
StorageContext is a higher-level abstraction that bundles together all storage components
needed for indexing and querying—including the vector_store, document store, index store,
and others. It manages how and where all types of data (not just embeddings) are stored 
and persisted, and is passed to index or pipeline objects for unified storage configuration
"""
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# 5.0
# Build and persist the index from StorageContext
# Pass here documents to storage_context
index = VectorStoreIndex.from_documents(
                                        documents, 
                                        storage_context=storage_context
                                        )


2026-08-25 14:30:49,602 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-25 14:30:54,688 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-25 14:30:58,983 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-25 14:31:02,711 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


In [ ]:
# 6.0

query_engine = index.as_query_engine()
response = query_engine.query("What is the main topic of sports file?")
print(response)



2026-08-25 14:26:15,645 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-08-25 14:26:15,683 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-25 14:26:47,868 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


 The main topic of the provided sports file is "Physical Activity and Sports—Real Health Benefits: A Review with Insight into the Public Health of Sweden". This document discusses the various health benefits and potential negative effects associated with sports, particularly in the context of Swedish public health. It explores both the physiological and psychosocial health benefits stemming from physical activity and sport participation. The file also highlights the importance of organized exercise and training in today's less physically active lifestyle, and the role sports play in preventing or alleviating mental illness.


In [ ]:
# 7.0
response = query_engine.query("What is the main topic of metagpt file?")
print(response)

2026-08-24 15:41:47,121 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-24 15:41:57,797 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


 The main topic of the Metagpt file appears to be the introduction and description of MetaGPT, a system for efficient meta-programming through a well-organized group of specialized agents. The paper discusses how MetaGPT utilizes software engineering SOPs to decompose complex tasks into actionable procedures, increase the success rate of target code generation, and minimize ambiguities and errors during collaboration. It also mentions that MetaGPT uses publicly available evaluation tools like HumanEval and MBPP for validation and outperforms other popular frameworks in handling software complexity and functionality.


In [ ]:
###############3